In [1]:
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# MOdel

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

#StrOutputParser

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
prompt=ChatPromptTemplate.from_template(
    """
    Explain {topic} in simple language
    """
)

In [6]:
parser=StrOutputParser()

এটা Response কে

String

এ convert করবে।

---



In [7]:
chain=prompt | llm | parser

In [8]:
response=chain.invoke(
    {
        "topic":"Neural Network"
    }
)
print(response)

Imagine your brain, but a super simplified, artificial version of it inside a computer. That's essentially a Neural Network!

Here's a breakdown:

1.  **Inspired by the Brain:** Our brains are made of billions of tiny cells called **neurons** that are connected to each other. They communicate by sending electrical signals. When you learn something new, these connections strengthen or weaken. An Artificial Neural Network tries to mimic this structure and learning process.

2.  **The "Neurons" (Nodes):**
    *   Instead of biological cells, a Neural Network has **artificial neurons** (often called **nodes** or **units**).
    *   These nodes are organized into layers:
        *   **Input Layer:** This is where you feed in your data. Think of it like your senses. If you're showing the network a picture of a cat, each node in the input layer might represent a single pixel's brightness or color.
        *   **Hidden Layers:** These are the "thinking" layers in between. This is where the mag

 we dont need to print(response.content) because parser return the string so we just print(response)


---



# Json Output

In [9]:
prompt=ChatPromptTemplate.from_template(
    """
    Return JSON
    Topic:{topic}
    keys
    title
    difficulty
    summary
    """
)

In [10]:
chain=prompt | llm | parser
response=chain.invoke(
    {
        "topic":"Neural Netwrok"
    }
)
print(response)

```json
{
  "title": "Introduction to Neural Networks: The Foundation of Modern AI",
  "difficulty": "Intermediate",
  "summary": "Neural Networks are a subset of machine learning, inspired by the structure and function of the human brain. They consist of interconnected nodes (neurons) organized in layers (input, hidden, output). Each connection between neurons has a weight, and each neuron has an activation function. During training, the network learns to adjust these weights by processing large amounts of data, minimizing the difference between its predictions and the actual outcomes. This process allows them to recognize complex patterns, classify data, make predictions, and drive advancements in fields like computer vision, natural language processing, and autonomous systems."
}
```


#JsonOutputParser

In [11]:
from langchain_core.output_parsers import JsonOutputParser

In [12]:
parser=JsonOutputParser()

In [13]:
print(parser.get_format_instructions())

Return a JSON object.


In [20]:
prompt=ChatPromptTemplate.from_template(
    """
    Return Only Valid JSON
    {format_instructions}
    Topic:{topic}
    keys
    title
    difficulty
    summary
    """
).partial(
    format_instructions=parser.get_format_instructions()
)


In [21]:
prompt

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={'format_instructions': 'Return a JSON object.'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['format_instructions', 'topic'], input_types={}, partial_variables={}, template='\n    Return Only Valid JSON\n    {format_instructions}\n    Topic:{topic}\n    keys\n    title\n    difficulty\n    summary\n    '), additional_kwargs={})])

In [22]:
chain=prompt | llm | parser

In [23]:
response=chain.invoke(
    {
        'topic':'Neural Network'
    }
)
print(response)

{'title': 'Introduction to Neural Networks', 'difficulty': 'Advanced', 'summary': "Neural Networks are a powerful subset of machine learning, inspired by the structure and function of the human brain. They consist of interconnected nodes (neurons) organized into layers (input, hidden, and output) that process information. Through a training process with large datasets, these networks learn to recognize patterns, classify data, and make predictions by adjusting the 'weights' of connections between neurons. They are fundamental to deep learning and are widely applied in fields like image recognition, natural language processing, and predictive analytics."}


In [25]:
response

{'title': 'Introduction to Neural Networks',
 'difficulty': 'Advanced',
 'summary': "Neural Networks are a powerful subset of machine learning, inspired by the structure and function of the human brain. They consist of interconnected nodes (neurons) organized into layers (input, hidden, and output) that process information. Through a training process with large datasets, these networks learn to recognize patterns, classify data, and make predictions by adjusting the 'weights' of connections between neurons. They are fundamental to deep learning and are widely applied in fields like image recognition, natural language processing, and predictive analytics."}

#PydanticOutputParser

In [26]:
from pydantic import BaseModel,Field

##schema

In [28]:
class Course(BaseModel):
  title: str=(
      Field(
          description="Course Name"
      )
  )
  duration: str
  level:str

In [29]:
course = Course(
    title="Machine Learning",
    duration="12 Weeks",
    level="Beginner"
)

print(course)

title='Machine Learning' duration='12 Weeks' level='Beginner'


In [30]:
from langchain_core.output_parsers import PydanticOutputParser

In [31]:
parser=PydanticOutputParser(pydantic_object=Course)

In [32]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "Course Name", "title": "Title", "type": "string"}, "duration": {"title": "Duration", "type": "string"}, "level": {"title": "Level", "type": "string"}}, "required": ["title", "duration", "level"]}
```


In [36]:
prompt=ChatPromptTemplate.from_template(
    """
    {format}
    topic : {topic}
    """
).partial(
    format=parser.get_format_instructions()
)

In [37]:
chain=prompt |llm | parser

In [38]:
response=chain.invoke(
    {
        "topic":"Machine Learning"
    }
)

In [39]:
response

Course(title='Introduction to Machine Learning', duration='10 weeks', level='Beginner')

In [40]:
print(response.title)
print(response.duration)

Introduction to Machine Learning
10 weeks
